In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import cv2

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
import os 
os.listdir('data_small/dataset/train')

['no_watermark', 'watermark']

In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
def takeFileName(filedir): # remove just file name from directory and return
    # filename = np.array(filedir.split('/'))[-1].split('.')[0] # take out the name, isolate the jpeg, then return the name
    filename = np.array(filedir.split('/'))[-1] # take out the name, then return the name
    # print(filename)
    return filename

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
train_path_watermarked_images = 'data_small/dataset/train/watermark/'
train_path_nonwatermarked_images = 'data_small/dataset/train/no_watermark/'

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
tp_watermarked = np.array([]) # array with watermarked image names
tp_nonwatermarked = np.array([]) # array with nonwatermarked image names

for root, dirs, files in os.walk(train_path_watermarked_images, topdown=True): # data length = 12510
    for file in files:
        tp_watermarked = np.append(tp_watermarked, takeFileName(file)) # append just the name of the file into array
    
for root, dirs, files in os.walk(train_path_nonwatermarked_images, topdown=True): # data length = 12477
    for file in files:
        tp_nonwatermarked = np.append(tp_nonwatermarked, takeFileName(file)) # append just the name of the file into array

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
output_array_wm = []

for i in tp_watermarked:
    output_string_wm = train_path_watermarked_images + i
    output_array_wm.append(output_string_wm)
    out_array_wm=np.array(output_array_wm)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
output_array_nwm = []

for i in tp_nonwatermarked:
    output_string_nwm = train_path_nonwatermarked_images + i
    output_array_nwm.append(output_string_nwm)
    out_array_nwm=np.array(output_array_nwm)

In [8]:
# --- [CELL 7]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# dimension to resize to 
width = 196 # only certain dimensions work due to UpSampling (196x196 works, 148x148 works)
height = 196
dim = (width, height) # set the dimensions
def createPixelArr(files):
    data = []
    for image in files:
        try: # take each image and use imread to get the pixel values in a matrix 
            img_arr = cv2.imread(image, cv2.IMREAD_COLOR)
            img_arr = cv2.cvtColor(img_arr, cv2.COLOR_BGR2RGB)
            resized_arr = cv2.resize(img_arr, (width, height)) # rescale the image so every image is of the same dimension
            data.append(resized_arr) # add the matrix of pixel values 
        except Exception as e:
            print(e) # some error thrown in imread or resize
    return np.array(data)

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
train_wms_pixVals = createPixelArr(out_array_wm[:90]) # 1000
train_nwms_pixVals = createPixelArr(out_array_nwm[:90]) # 1000

In [10]:
# --- [CELL 9]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 10}
X_train, X_test, y_train, y_test = train_test_split(train_wms_pixVals, train_nwms_pixVals, train_size=0.8, random_state=1)

In [11]:
# --- [CELL 10]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 11}
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import time
import os
import copy
from PIL import Image
import pandas as pd
import random
from tqdm import tqdm
import timm
import torch.nn.functional as F
from timm.models.layers import trunc_normal_, DropPath
from timm.models.registry import register_model

/usr/local/lib/python3.10/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/usr/local/lib/python3.10/dist-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [12]:
# --- [CELL 11]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 12}
model_ft = timm.create_model(
    'efficientnet_b3a', pretrained=True, num_classes=2
)
model_ft.classifier = nn.Sequential(
    nn.Linear(in_features=1536, out_features=625),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(in_features=625, out_features=256),
    nn.ReLU(),
    nn.Linear(in_features=256, out_features=2),
)

/usr/local/lib/python3.10/dist-packages/timm/models/_factory.py:117: UserWarning: Mapping deprecated model name efficientnet_b3a to current efficientnet_b3.
  model = create_fn(


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

In [13]:
# --- [CELL 12]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 13}
# === BEFORE (original) ===
# device = torch.device('cpu') # 'cuda:0'
# 
# def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=80):
#     since = time.time()
# 
#     val_acc_history = []
#     train_acc_history = []
# 
#     best_model_wts = copy.deepcopy(model.state_dict())
#     best_acc = 0.0
# 
#     for epoch in range(num_epochs):
#         print('Epoch {}/{}'.format(epoch, num_epochs - 1))
#         print('-' * 10)
# 
#         model.train()
# 
#         running_loss = 0.0
#         running_corrects = 0
# 
#         for inputs, labels in tqdm(train_loader):
#             inputs = inputs.to(device)
#             labels = labels.to(device)
# 
#             optimizer.zero_grad()
# 
#             with torch.set_grad_enabled(True):
#                 with torch.cuda.amp.autocast():
#                     outputs = model(inputs)
#                     loss = criterion(outputs, labels)
# 
#                 _, preds = torch.max(outputs, 1)
# 
#                 loss.backward()
#                 optimizer.step()
# 
#             running_loss += loss.item() * inputs.size(0)
#             running_corrects += torch.sum(preds == labels.data)
# 
#         epoch_loss = running_loss / len(train_loader.dataset)
#         epoch_acc = running_corrects.double() / len(train_loader.dataset)
# 
#         print('Train Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss, epoch_acc))
#         train_acc_history.append(epoch_acc)
# 
#         model.eval()
# 
#         running_loss = 0.0
#         running_corrects = 0
# 
#         for inputs, labels in tqdm(test_loader):
#             inputs = inputs.to(device)
#             labels = labels.to(device)
# 
#             with torch.set_grad_enabled(False):
#                 with torch.cuda.amp.autocast():
#                     outputs = model(inputs)
#                     loss = criterion(outputs, labels)
# 
#                 _, preds = torch.max(outputs, 1)
# 
#             running_loss += loss.item() * inputs.size(0)
#             running_corrects += torch.sum(preds == labels.data)
# 
#         epoch_loss = running_loss / len(test_loader.dataset)
#         epoch_acc = running_corrects.double() / len(test_loader.dataset)
# 
#         print('Test Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss, epoch_acc))
#         val_acc_history.append(epoch_acc)
# 
#         if epoch_acc > best_acc:
#             best_acc = epoch_acc
#             best_model_wts = copy.deepcopy(model.state_dict())
# 
#         print()
# 
#     time_elapsed = time.time() - since
#     print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
#     print('Best val Acc: {:4f}'.format(best_acc))
# 
#     model.load_state_dict(best_model_wts)
#     return model, train_acc_history, val_acc_history

# === AFTER (edited) ===
device = torch.device('cpu')

def train_model(model, train_loader, test_loader, criterion, optimizer, num_epochs=80):
    since = time.time()

    val_acc_history = []
    train_acc_history = []

    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        model.train()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in tqdm(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            with torch.set_grad_enabled(True):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

                loss.backward()
                optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = running_corrects.double() / len(train_loader.dataset)

        print('Train Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss, epoch_acc))
        train_acc_history.append(epoch_acc)

        model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in tqdm(test_loader):
            inputs = inputs.to(device)
            labels = labels.to(device)

            with torch.set_grad_enabled(False):
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                _, preds = torch.max(outputs, 1)

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(test_loader.dataset)
        epoch_acc = running_corrects.double() / len(test_loader.dataset)

        print('Test Loss: {:.4f} Acc: {:.4f}'.format(epoch_loss, epoch_acc))
        val_acc_history.append(epoch_acc)

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print('Training complete in {:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))
    print('Best val Acc: {:4f}'.format(best_acc))

    model.load_state_dict(best_model_wts)
    return model, train_acc_history, val_acc_history

In [14]:
# --- [CELL 13]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 14}
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.AdamW(params=model_ft.parameters(), lr=0.2e-5)

In [15]:
# --- [CELL 14]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 15}
class MyDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [16]:
# --- [CELL 15]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 16}
train_dataset = MyDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

In [17]:
# --- [CELL 16]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 17}
test_dataset = MyDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [18]:
# --- [CELL 17]: ---
# cell_state: unchanged
# execution_status: {'status': 'error', 'done': True, 'execution_count': 18}
import warnings
warnings.filterwarnings("ignore")

model_ft, train_acc_history, val_acc_history = train_model(
    model_ft, train_loader, test_loader, criterion, optimizer, num_epochs=3
)

Epoch 0/2
----------


  0%|          | 0/2 [00:00<?, ?it/s]


RuntimeError: Given groups=1, weight of size [40, 3, 3, 3], expected input[64, 196, 196, 3] to have 3 channels, but got 196 channels instead

In [19]:
# 1) MyDataset must emit NCHW float32 inputs + long labels
# 2) Label construction should represent two classes (0/1)
# 3) train_model path should produce valid metric histories

import torch

# ----- dataset tensor format/types -----
assert "train_loader" in globals(), "train_loader missing"
xb, yb = next(iter(train_loader))

assert isinstance(xb, torch.Tensor) and isinstance(yb, torch.Tensor), "Loader must return tensors"
assert xb.ndim == 4, f"Expected input batch shape [N, C, H, W], got {tuple(xb.shape)}"
assert xb.shape[1] == 3, f"Expected channels-first RGB inputs with C=3, got C={xb.shape[1]}"
assert xb.dtype == torch.float32, f"Expected input dtype float32, got {xb.dtype}"

assert yb.ndim == 1, f"Expected 1D class-index labels, got shape {tuple(yb.shape)}"
assert yb.dtype == torch.long, f"Expected label dtype torch.long, got {yb.dtype}"
assert xb.shape[0] == yb.shape[0], "Input/label batch sizes must match"

# ----- binary label construction sanity -----
assert "y_train" in globals() and "y_test" in globals(), "Expected y_train/y_test from split"
y_train_unique = set(torch.tensor(y_train).long().unique().tolist())
y_test_unique = set(torch.tensor(y_test).long().unique().tolist())

assert y_train_unique.issubset({0, 1}), f"y_train has non-binary labels: {y_train_unique}"
assert y_test_unique.issubset({0, 1}), f"y_test has non-binary labels: {y_test_unique}"
assert len(y_train_unique | y_test_unique) == 2, "Expected both classes {0,1} to be present"

# ----- training loop dtype/label compatibility + history quality -----
assert "train_acc_history" in globals(), "train_acc_history missing; training likely failed"
assert "val_acc_history" in globals(), "val_acc_history missing; training likely failed"
assert len(train_acc_history) == 3, f"Expected 3 train epochs, got {len(train_acc_history)}"
assert len(val_acc_history) == 3, f"Expected 3 validation epochs, got {len(val_acc_history)}"

train_vals = [float(x.detach().cpu()) if hasattr(x, "detach") else float(x) for x in train_acc_history]
val_vals = [float(x.detach().cpu()) if hasattr(x, "detach") else float(x) for x in val_acc_history]

assert all(0.0 <= v <= 1.0 for v in train_vals), f"Train accuracies out of range: {train_vals}"
assert all(0.0 <= v <= 1.0 for v in val_vals), f"Validation accuracies out of range: {val_vals}"

# Forward + CE contract on one batch (catches common dtype/shape regressions)
assert "model_ft" in globals() and "criterion" in globals(), "model_ft/criterion missing"
model_ft.eval()
with torch.no_grad():
    logits = model_ft(xb.to(device).float())
    loss_val = criterion(logits, yb.to(device).long())

assert logits.ndim == 2 and logits.shape[0] == yb.shape[0] and logits.shape[1] == 2, (
    f"Expected logits shape [N,2], got {tuple(logits.shape)}"
)
assert torch.isfinite(logits).all().item(), "Logits contain NaN/Inf"
assert torch.isfinite(loss_val).item(), "CrossEntropyLoss is not finite"

AssertionError: Expected channels-first RGB inputs with C=3, got C=196